# Factor Model Backtesting

## Objective
Test factor model performance using long-short portfolios.

### Strategy:
For each factor:
1. Sort stocks by factor loading (from Step 4)
2. Go **long** top decile (highest 10%)
3. Go **short** bottom decile (lowest 10%)
4. Rebalance monthly
5. Calculate portfolio returns using actual stock returns

### Performance Metrics:
- **Sharpe Ratio**: Risk-adjusted return
- **Cumulative Return**: Total return over period
- **Max Drawdown**: Largest peak-to-trough decline
- **Win Rate**: Percentage of positive return days
- **Information Ratio**: vs Russell 2000 benchmark

### Inputs:
- `russell2000_factor_loadings_step4.parquet` - Factor loadings (B matrix)
- `russell2000_winsorized_step3.parquet` - Actual returns
- `factor_returns_step5.parquet` - Factor returns (optional)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Load Data

In [ ]:
print("="*80)
print("LOADING DATA")
print("="*80)

# Load factor loadings
print("\nLoading factor loadings...")
df_loadings = pd.read_parquet('russell2000_factor_loadings_step4.parquet')

print(f"✓ Factor loadings loaded")
print(f"  Shape: {df_loadings.shape}")
print(f"  Factors: {len(df_loadings.columns)}")
print(f"  Date range: {df_loadings.index.get_level_values(0).min()} to {df_loadings.index.get_level_values(0).max()}")

# Load returns
print("\nLoading winsorized returns...")
df_returns = pd.read_parquet('russell2000_winsorized_step3.parquet')
returns = df_returns['return_winsorized']

print(f"✓ Returns loaded")
print(f"  Shape: {returns.shape}")
print(f"  Date range: {returns.index.get_level_values(0).min()} to {returns.index.get_level_values(0).max()}")

# Align indices
print("\nAligning indices...")
common_index = df_loadings.index.intersection(returns.index)
df_loadings = df_loadings.loc[common_index]
returns = returns.loc[common_index]

print(f"✓ Data aligned")
print(f"  Common observations: {len(common_index):,}")
print(f"  Dates: {common_index.get_level_values(0).nunique():,}")
print(f"  Stocks: {common_index.get_level_values(1).nunique():,}")

## 2. Backtesting Parameters

In [ ]:
print("="*80)
print("BACKTESTING PARAMETERS")
print("="*80)

# Strategy parameters
REBALANCE_FREQ = 'M'  # 'D' = daily, 'W' = weekly, 'M' = monthly
TOP_DECILE = 0.1      # Top 10% for long
BOTTOM_DECILE = 0.1   # Bottom 10% for short
HOLDING_PERIOD = 1    # Days to hold after rebalance

# Transaction costs (optional)
TRANSACTION_COST_BPS = 10  # 10 basis points per trade

# Number of top factors to backtest (for speed)
N_FACTORS_TO_TEST = 50  # Test top 50 factors

print(f"\nStrategy:")
print(f"  Rebalance frequency: {REBALANCE_FREQ} ({'Monthly' if REBALANCE_FREQ == 'M' else 'Daily' if REBALANCE_FREQ == 'D' else 'Weekly'})")
print(f"  Long: Top {TOP_DECILE*100:.0f}% by factor loading")
print(f"  Short: Bottom {BOTTOM_DECILE*100:.0f}% by factor loading")
print(f"  Transaction cost: {TRANSACTION_COST_BPS} bps per trade")

print(f"\nTesting: Top {N_FACTORS_TO_TEST} factors (by variance)")

## 3. Select Factors to Test

Test factors with highest variance (most signal)

In [ ]:
print("="*80)
print("SELECTING FACTORS TO TEST")
print("="*80)

# Calculate factor variance (cross-sectional)
factor_vars = df_loadings.groupby(level=0).var().mean()

# Select top N factors by variance
top_factors = factor_vars.nlargest(N_FACTORS_TO_TEST).index.tolist()

print(f"\n✓ Selected {len(top_factors)} factors")
print(f"\nTop 10 factors by variance:")
for i, factor in enumerate(top_factors[:10], 1):
    print(f"  {i:2d}. {factor:30s} (var: {factor_vars[factor]:.6f})")

## 4. Run Long-Short Backtest

For each factor, create long-short portfolio and calculate returns.

In [ ]:
def backtest_factor_long_short(factor_name, df_loadings, returns, 
                                 top_pct=0.1, bottom_pct=0.1, 
                                 rebalance_freq='M'):
    """
    Backtest a single factor using long-short strategy.
    
    Strategy:
    1. Rank stocks by factor loading
    2. Long top decile, Short bottom decile
    3. Equal-weighted within each leg
    4. Rebalance periodically
    
    Returns:
    --------
    portfolio_returns : pd.Series
        Daily returns of the long-short portfolio
    """
    
    # Extract factor loadings
    factor_loadings = df_loadings[factor_name]
    
    # Get unique dates
    dates = factor_loadings.index.get_level_values(0).unique().sort_values()
    
    # Rebalance dates
    if rebalance_freq == 'M':
        rebalance_dates = dates.to_series().resample('M').last().values
    elif rebalance_freq == 'W':
        rebalance_dates = dates.to_series().resample('W').last().values
    else:
        rebalance_dates = dates
    
    portfolio_returns = []
    current_long = set()
    current_short = set()
    
    for i, date in enumerate(dates):
        # Rebalance if needed
        if date in rebalance_dates:
            try:
                # Get factor loadings for this date
                loadings_t = factor_loadings.xs(date, level=0).dropna()
                
                if len(loadings_t) < 20:  # Need minimum stocks
                    continue
                
                # Rank stocks by loading
                loadings_sorted = loadings_t.sort_values(ascending=False)
                
                # Select long (top decile) and short (bottom decile)
                n_long = max(1, int(len(loadings_sorted) * top_pct))
                n_short = max(1, int(len(loadings_sorted) * bottom_pct))
                
                current_long = set(loadings_sorted.head(n_long).index)
                current_short = set(loadings_sorted.tail(n_short).index)
                
            except Exception as e:
                pass
        
        # Calculate portfolio return for this date
        try:
            returns_t = returns.xs(date, level=0)
            
            # Long leg return
            long_stocks = list(current_long.intersection(returns_t.index))
            long_return = returns_t.loc[long_stocks].mean() if long_stocks else 0
            
            # Short leg return  
            short_stocks = list(current_short.intersection(returns_t.index))
            short_return = returns_t.loc[short_stocks].mean() if short_stocks else 0
            
            # Portfolio return: 0.5 * long - 0.5 * short (dollar neutral)
            portfolio_return = 0.5 * long_return - 0.5 * short_return
            
            portfolio_returns.append({
                'date': date,
                'return': portfolio_return,
                'n_long': len(long_stocks),
                'n_short': len(short_stocks)
            })
            
        except Exception as e:
            continue
    
    # Convert to DataFrame
    df_results = pd.DataFrame(portfolio_returns)
    df_results = df_results.set_index('date')
    
    return df_results['return']


print("="*80)
print("RUNNING BACKTESTS")
print("="*80)
print(f"\nTesting {len(top_factors)} factors...\n")

# Run backtest for each factor
backtest_results = {}

for factor in tqdm(top_factors, desc="Backtesting factors"):
    try:
        portfolio_returns = backtest_factor_long_short(
            factor, df_loadings, returns,
            top_pct=TOP_DECILE,
            bottom_pct=BOTTOM_DECILE,
            rebalance_freq=REBALANCE_FREQ
        )
        
        if len(portfolio_returns) > 0:
            backtest_results[factor] = portfolio_returns
            
    except Exception as e:
        print(f"\n⚠ Error with {factor}: {e}")
        continue

print(f"\n✓ Backtest complete")
print(f"  Successful: {len(backtest_results)}/{len(top_factors)} factors")

## 5. Calculate Performance Metrics

In [ ]:
def calculate_performance_metrics(returns_series):
    """
    Calculate comprehensive performance metrics.
    """
    returns = returns_series.dropna()
    
    if len(returns) == 0:
        return {}
    
    # Basic statistics
    total_return = (1 + returns).prod() - 1
    annualized_return = (1 + total_return) ** (252 / len(returns)) - 1
    annualized_vol = returns.std() * np.sqrt(252)
    sharpe_ratio = annualized_return / annualized_vol if annualized_vol > 0 else 0
    
    # Drawdown
    cum_returns = (1 + returns).cumprod()
    running_max = cum_returns.cummax()
    drawdown = (cum_returns - running_max) / running_max
    max_drawdown = drawdown.min()
    
    # Win rate
    win_rate = (returns > 0).sum() / len(returns)
    
    # Calmar ratio (return / max drawdown)
    calmar_ratio = abs(annualized_return / max_drawdown) if max_drawdown < 0 else 0
    
    return {
        'Total Return': total_return,
        'Annual Return': annualized_return,
        'Annual Volatility': annualized_vol,
        'Sharpe Ratio': sharpe_ratio,
        'Max Drawdown': max_drawdown,
        'Calmar Ratio': calmar_ratio,
        'Win Rate': win_rate,
        'Number of Days': len(returns)
    }


print("="*80)
print("CALCULATING PERFORMANCE METRICS")
print("="*80)

# Calculate metrics for all factors
performance_metrics = {}

for factor, returns in backtest_results.items():
    metrics = calculate_performance_metrics(returns)
    performance_metrics[factor] = metrics

# Convert to DataFrame
df_performance = pd.DataFrame(performance_metrics).T

# Sort by Sharpe ratio
df_performance = df_performance.sort_values('Sharpe Ratio', ascending=False)

print(f"\n✓ Performance metrics calculated for {len(df_performance)} factors")
print(f"\nTop 10 Factors by Sharpe Ratio:")
display(df_performance.head(10))

print(f"\nBottom 10 Factors by Sharpe Ratio:")
display(df_performance.tail(10))

## 6. Visualization

In [ ]:
print("="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

# Plot 1: Cumulative returns of top 10 factors
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top 10 cumulative returns
top_10_factors = df_performance.head(10).index
for factor in top_10_factors:
    cum_returns = (1 + backtest_results[factor]).cumprod()
    axes[0, 0].plot(cum_returns.index, cum_returns.values, label=factor, linewidth=1.5, alpha=0.7)

axes[0, 0].axhline(1, color='black', linestyle='--', linewidth=1)
axes[0, 0].set_title('Cumulative Returns: Top 10 Factors by Sharpe', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Cumulative Return')
axes[0, 0].legend(loc='best', fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# Sharpe ratio distribution
axes[0, 1].hist(df_performance['Sharpe Ratio'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[0, 1].axvline(df_performance['Sharpe Ratio'].median(), color='green', linestyle='--', 
                   linewidth=2, label=f"Median: {df_performance['Sharpe Ratio'].median():.2f}")
axes[0, 1].set_title('Distribution of Sharpe Ratios', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Sharpe Ratio')
axes[0, 1].set_ylabel('Number of Factors')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Return vs Volatility scatter
axes[1, 0].scatter(df_performance['Annual Volatility'], df_performance['Annual Return'], 
                   alpha=0.6, s=50)
axes[1, 0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1, 0].set_title('Risk-Return Profile', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Annual Volatility')
axes[1, 0].set_ylabel('Annual Return')
axes[1, 0].grid(True, alpha=0.3)

# Max drawdown distribution
axes[1, 1].hist(df_performance['Max Drawdown'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(df_performance['Max Drawdown'].median(), color='green', linestyle='--', 
                   linewidth=2, label=f"Median: {df_performance['Max Drawdown'].median():.2%}")
axes[1, 1].set_title('Distribution of Maximum Drawdowns', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Max Drawdown')
axes[1, 1].set_ylabel('Number of Factors')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('backtest_performance.png', dpi=150, bbox_inches='tight')
print("\n✓ Saved: backtest_performance.png")
plt.show()

## 7. Best Factor Deep Dive

In [ ]:
print("="*80)
print("BEST FACTOR ANALYSIS")
print("="*80)

best_factor = df_performance.index[0]
best_returns = backtest_results[best_factor]

print(f"\nBest Factor: {best_factor}")
print(f"\nMetrics:")
for metric, value in df_performance.loc[best_factor].items():
    if isinstance(value, float):
        if 'Rate' in metric or 'Return' in metric or 'Drawdown' in metric:
            print(f"  {metric:20s}: {value:>10.2%}")
        else:
            print(f"  {metric:20s}: {value:>10.4f}")
    else:
        print(f"  {metric:20s}: {value:>10}")

# Plot detailed analysis
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Cumulative returns
cum_returns = (1 + best_returns).cumprod()
axes[0].plot(cum_returns.index, cum_returns.values, linewidth=2, color='darkblue')
axes[0].axhline(1, color='black', linestyle='--', linewidth=1, label='Baseline')
axes[0].fill_between(cum_returns.index, 1, cum_returns.values, alpha=0.3)
axes[0].set_title(f'Cumulative Returns: {best_factor}', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cumulative Return')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Drawdown
running_max = cum_returns.cummax()
drawdown = (cum_returns - running_max) / running_max
axes[1].fill_between(drawdown.index, 0, drawdown.values, color='red', alpha=0.5)
axes[1].plot(drawdown.index, drawdown.values, color='darkred', linewidth=1.5)
axes[1].set_title('Drawdown Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Drawdown')
axes[1].grid(True, alpha=0.3)

# Rolling Sharpe (90-day)
rolling_sharpe = best_returns.rolling(90).mean() / best_returns.rolling(90).std() * np.sqrt(252)
axes[2].plot(rolling_sharpe.index, rolling_sharpe.values, linewidth=2, color='darkgreen')
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].axhline(rolling_sharpe.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {rolling_sharpe.mean():.2f}')
axes[2].set_title('Rolling 90-Day Sharpe Ratio', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Sharpe Ratio')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('best_factor_analysis.png', dpi=150, bbox_inches='tight')
print("\n✓ Saved: best_factor_analysis.png")
plt.show()

## 8. Save Results

In [ ]:
print("="*80)
print("SAVING RESULTS")
print("="*80)

# Save performance metrics
df_performance.to_csv('backtest_performance_metrics.csv')
print("\n✓ Saved: backtest_performance_metrics.csv")

# Save all portfolio returns
df_all_returns = pd.DataFrame(backtest_results)
df_all_returns.to_parquet('backtest_all_returns.parquet')
print("✓ Saved: backtest_all_returns.parquet")

# Save summary statistics
summary = {
    'Total Factors Tested': len(top_factors),
    'Successful Backtests': len(backtest_results),
    'Median Sharpe Ratio': df_performance['Sharpe Ratio'].median(),
    'Mean Sharpe Ratio': df_performance['Sharpe Ratio'].mean(),
    'Positive Sharpe Factors': (df_performance['Sharpe Ratio'] > 0).sum(),
    'Best Factor': best_factor,
    'Best Sharpe': df_performance.loc[best_factor, 'Sharpe Ratio'],
    'Rebalance Frequency': REBALANCE_FREQ,
    'Date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

df_summary = pd.Series(summary)
df_summary.to_csv('backtest_summary.csv', header=['Value'])
print("✓ Saved: backtest_summary.csv")

print("\n" + "="*80)
print("✅ BACKTEST COMPLETE")
print("="*80)

## 9. Summary Report

In [ ]:
print("="*80)
print("BACKTEST SUMMARY REPORT")
print("="*80)

print(f"\nBacktest Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "="*80)
print("STRATEGY")
print("="*80)
print(f"\n  Type: Long-Short (Decile)")
print(f"  Long: Top {TOP_DECILE*100:.0f}% by factor loading")
print(f"  Short: Bottom {BOTTOM_DECILE*100:.0f}% by factor loading")
print(f"  Rebalance: {REBALANCE_FREQ} ({'Monthly' if REBALANCE_FREQ == 'M' else 'Daily' if REBALANCE_FREQ == 'D' else 'Weekly'})")
print(f"  Position: Equal-weighted within each leg")

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
print(f"\n  Factors tested: {len(top_factors)}")
print(f"  Successful: {len(backtest_results)}")
print(f"  Positive Sharpe: {(df_performance['Sharpe Ratio'] > 0).sum()} ({(df_performance['Sharpe Ratio'] > 0).sum() / len(df_performance) * 100:.1f}%)")

print(f"\n  Sharpe Ratios:")
print(f"    Mean: {df_performance['Sharpe Ratio'].mean():.4f}")
print(f"    Median: {df_performance['Sharpe Ratio'].median():.4f}")
print(f"    Best: {df_performance['Sharpe Ratio'].max():.4f} ({df_performance['Sharpe Ratio'].idxmax()})")
print(f"    Worst: {df_performance['Sharpe Ratio'].min():.4f} ({df_performance['Sharpe Ratio'].idxmin()})")

print(f"\n  Annual Returns:")
print(f"    Mean: {df_performance['Annual Return'].mean():.2%}")
print(f"    Median: {df_performance['Annual Return'].median():.2%}")

print(f"\n  Max Drawdowns:")
print(f"    Mean: {df_performance['Max Drawdown'].mean():.2%}")
print(f"    Median: {df_performance['Max Drawdown'].median():.2%}")

print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)

mean_sharpe = df_performance['Sharpe Ratio'].mean()
pct_positive = (df_performance['Sharpe Ratio'] > 0).sum() / len(df_performance) * 100

if mean_sharpe > 0.5 and pct_positive > 60:
    print("\n  ✅ STRONG RESULTS")
    print("     - Many factors have positive risk-adjusted returns")
    print("     - Factor model appears to capture real signal")
    print("     - Consider further optimization and out-of-sample testing")
elif mean_sharpe > 0 and pct_positive > 50:
    print("\n  ⚠️  MODERATE RESULTS")
    print("     - Some factors work, but results are mixed")
    print("     - May need factor selection or better combination")
    print("     - Consider transaction costs impact")
else:
    print("\n  ❌ WEAK RESULTS")
    print("     - Most factors have poor performance")
    print("     - Model may not capture meaningful signal")
    print("     - Consider: data quality, factor construction, or overfitting")

print("\n" + "="*80)
print("OUTPUT FILES")
print("="*80)
print("\n  ✓ backtest_performance_metrics.csv - All factor metrics")
print("  ✓ backtest_all_returns.parquet - Daily returns for all factors")
print("  ✓ backtest_summary.csv - Summary statistics")
print("  ✓ backtest_performance.png - Performance visualizations")
print("  ✓ best_factor_analysis.png - Best factor deep dive")

print("\n" + "="*80)
print("NEXT STEPS")
print("="*80)
print("\n  1. Review top factors - why do they work?")
print("  2. Check for overfitting - test on recent out-of-sample period")
print("  3. Consider combining multiple factors (portfolio optimization)")
print("  4. Estimate transaction costs impact")
print("  5. Run sensitivity analysis (different rebalance frequencies, etc.)")

print("\n" + "="*80)